# 02 - Feature Engineering: Molecular Descriptors

This notebook demonstrates the computation of molecular descriptors using RDKit.

**Descriptors computed:**
- Physicochemical properties (MW, LogP, TPSA, etc.)
- Morgan (ECFP) fingerprints
- MACCS keys
- Topological descriptors
- All RDKit descriptors

## 1. Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw

from src.features.descriptors import DescriptorGenerator
from src.features.selection import FeatureSelector
from src.utils.config import settings

print("Setup complete")

## 2. Load Data

In [ ]:
# Load collected data
data_path = settings.RAW_DATA_DIR / 'pubchem_compounds.csv'

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"Loaded {len(df)} compounds")
else:
    print(f"Data file not found at {data_path}")
    print("Run 01_data_collection.ipynb first")
    # Create sample data for demonstration
    df = pd.DataFrame({
        'smiles': ['CCO', 'CC(=O)O', 'c1ccccc1', 'CC(C)C', 'CCN'],
        'solubility': [1.0, 0.5, 0.01, 0.1, 0.8],
    })

df.head()

## 3. Compute Descriptors for a Single Molecule

In [ ]:
# Example: Ethanol
gen = DescriptorGenerator(
    morgan_radius=2,
    morgan_nbits=2048,
    include_maccs=True,
    include_topological=True,
    include_physicochemical=True,
)

smiles = 'CCO'  # Ethanol
descriptors = gen.compute_all(smiles)

print(f"Total descriptors computed: {len(descriptors)}")
print(f"\nPhysicochemical descriptors:")
physio_keys = ['MolWt', 'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors']
for k in physio_keys:
    print(f"  {k}: {descriptors.get(k, 'N/A')}")

print(f"\nMorgan fingerprint bits set: {sum(1 for k, v in descriptors.items() if k.startswith('Morgan_') and v)}")
print(f"MACCS keys set: {sum(1 for k, v in descriptors.items() if k.startswith('MACCS_') and v)}")

## 4. Compute Descriptors for the Entire Dataset

In [ ]:
# Compute descriptors for all compounds
smiles_list = df['smiles'].tolist()

print(f"Computing descriptors for {len(smiles_list)} molecules...")
descriptors_df = gen.compute_batch(smiles_list, show_progress=True)

print(f"\nDescriptors DataFrame shape: {descriptors_df.shape}")
print(f"Columns: {list(descriptors_df.columns[:20])}...")
descriptors_df.head()

## 5. Merge with Original Data

In [ ]:
# Merge descriptors with original data
descriptors_df = descriptors_df.drop(columns=['SMILES'], errors='ignore')
combined_df = pd.concat([df.reset_index(drop=True), descriptors_df], axis=1)

print(f"Combined DataFrame shape: {combined_df.shape}")
combined_df.head()

## 6. Feature Selection

In [ ]:
# Prepare features for selection
target_cols = ['solubility', 'boiling_point', 'toxicity_category']
target_cols = [c for c in target_cols if c in combined_df.columns]

feature_cols = [c for c in descriptors_df.columns
                if not c.startswith(('Morgan_', 'MACCS_'))]

X = combined_df[feature_cols].select_dtypes(include=[np.number])
X = X.dropna(axis=1)  # Drop columns with NaN

if target_cols:
    y = combined_df[target_cols[0]]  # Use first target
    
    print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")
    print(f"Target: {target_cols[0]}")
    
    # Apply feature selection
    selector = FeatureSelector(method='combined', n_features=50)
    X_selected = selector.select(X, y)
    
    print(f"\nSelected features: {X_selected.shape[1]}")
    print(f"Top 10 features:")
    for feat in selector.selected_features[:10]:
        print(f"  - {feat}")

## 7. Visualize Feature Importance

In [ ]:
if target_cols:
    importance_df = selector.get_feature_importance_df().head(15)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    importance_df = importance_df.sort_values('importance')
    ax.barh(importance_df['feature'], importance_df['importance'], color='#3498db')
    ax.set_xlabel('Importance Score')
    ax.set_title('Top 15 Feature Importance (Mutual Information)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 8. PCA Visualization

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardize and apply PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.2%}")

fig, ax = plt.subplots(figsize=(10, 8))
if target_cols:
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', alpha=0.6, s=50)
    plt.colorbar(scatter, ax=ax, label=target_cols[0])
else:
    ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, s=50, c='#3498db')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA Visualization of Molecular Descriptors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Save Features

In [ ]:
# Save processed dataset
output_path = settings.PROCESSED_DATA_DIR / 'features_selected.csv'

if target_cols:
    final_df = pd.concat([X_selected, y.rename(target_cols[0])], axis=1)
else:
    final_df = X_selected

final_df.to_csv(output_path, index=False)
print(f"Saved processed features to: {output_path}")
print(f"Shape: {final_df.shape}")

## 10. Summary

In this notebook, we:
1. Computed 2,000+ molecular descriptors using RDKit
2. Applied feature selection to identify the most informative descriptors
3. Visualized feature importance and PCA projections
4. Saved the processed feature matrix

**Next**: Go to `03_model_training.ipynb` to train ML models.